# Q1 coherence characterization ($T_1$ and Ramsey $T_2^*$)

This notebook produces the q1 coherence figure used in the Supplemental Material. The two portable bundles were copied from `data_opx1000/calibrations/2026-06-28`: `05_T1/17-16-39-684672` and `06a_ramsey/14-44-12-672280`. Each bundle retains the original metadata, profile snapshot, sweep, and measured state array.

The $T_1$ trace is fit to an exponential with an offset. Only the positive-detuning Ramsey trace is used, as in the Supplemental Material, and is fit to an exponentially damped sinusoid. Reported uncertainties are one-standard-deviation errors from the fit covariance matrix.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root.')


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'ips_plots/data/t12'
T1_PATH = DATA_DIR / '05_T1_17-16-39-684672_data_bundle.npz'
T2_PATH = DATA_DIR / '06a_ramsey_14-44-12-672280_data_bundle.npz'

import sys
sys.path.insert(0, str(PROJECT_ROOT))
from echospec.figures import FigureVariant, apply_figure_style, save_figure

VARIANT = FigureVariant.PAPER
apply_figure_style(VARIANT)

print(f'Project root: {PROJECT_ROOT}')
print(f'T1 data: {T1_PATH.relative_to(PROJECT_ROOT)}')
print(f'T2 data: {T2_PATH.relative_to(PROJECT_ROOT)}')

In [ ]:
def load_bundle(path):
    bundle = np.load(path, allow_pickle=True)
    metadata = json.loads(str(bundle['metadata.json'].item()))
    manifest = json.loads(str(bundle['bundle_manifest_json'].item()))
    return bundle, metadata, manifest


t1_bundle, t1_metadata, t1_manifest = load_bundle(T1_PATH)
t2_bundle, t2_metadata, t2_manifest = load_bundle(T2_PATH)

assert t1_bundle['data__qubit'].tolist() == ['q1']
assert t2_bundle['data__qubit'].tolist() == ['q1']

print('T1 source:', t1_manifest['source_path'])
print('T1 timestamp:', t1_metadata['timestamp'])
print('Ramsey source:', t2_manifest['source_path'])
print('Ramsey timestamp:', t2_metadata['timestamp'])

In [ ]:
def t1_decay(t_us, offset, amplitude, tau_us):
    return offset + amplitude * np.exp(-t_us / tau_us)


def ramsey_decay(t_us, offset, amplitude, tau_us, frequency_mhz, phase_rad):
    return offset + amplitude * np.exp(-t_us / tau_us) * np.cos(
        2 * np.pi * frequency_mhz * t_us + phase_rad
    )


def fit_t1(t_us, state):
    tail = np.mean(state[-10:])
    initial = [tail, state[0] - tail, 0.5 * np.ptp(t_us)]
    bounds = ([0.0, -1.2, 0.01], [1.2, 1.2, 10_000.0])
    optimum, covariance = curve_fit(
        t1_decay, t_us, state, p0=initial, bounds=bounds, maxfev=20_000
    )
    return optimum, np.sqrt(np.diag(covariance))


def fit_ramsey(t_us, state, frequency_guess_mhz=2.0):
    initial = [
        np.mean(state),
        0.5 * np.ptp(state),
        0.5 * np.ptp(t_us),
        frequency_guess_mhz,
        0.0,
    ]
    bounds = (
        [0.0, -1.0, 0.01, 0.0, -2 * np.pi],
        [1.2, 1.0, 10_000.0, 10.0, 2 * np.pi],
    )
    optimum, covariance = curve_fit(
        ramsey_decay, t_us, state, p0=initial, bounds=bounds, maxfev=100_000
    )
    return optimum, np.sqrt(np.diag(covariance))


def coefficient_of_determination(observed, fitted):
    residual_sum = np.sum((observed - fitted) ** 2)
    total_sum = np.sum((observed - np.mean(observed)) ** 2)
    return 1 - residual_sum / total_sum

In [ ]:
t1_time_us = t1_bundle['data__idle_time'] / 1_000
t1_state = t1_bundle['data__state'][0]
t1_fit, t1_error = fit_t1(t1_time_us, t1_state)
t1_fitted = t1_decay(t1_time_us, *t1_fit)
t1_r_squared = coefficient_of_determination(t1_state, t1_fitted)

ramsey_time_us = t2_bundle['data__idle_time'] / 1_000
ramsey_state_all = t2_bundle['data__state'][0]
detuning_signs = t2_bundle['data__detuning_signs']
positive_detuning_index = int(np.flatnonzero(detuning_signs == 1).item())
ramsey_state = ramsey_state_all[:, positive_detuning_index]
ramsey_fit, ramsey_error = fit_ramsey(ramsey_time_us, ramsey_state)
ramsey_fitted = ramsey_decay(ramsey_time_us, *ramsey_fit)
ramsey_r_squared = coefficient_of_determination(ramsey_state, ramsey_fitted)

print(f'T1 = {t1_fit[2]:.2f} +/- {t1_error[2]:.2f} us (R^2 = {t1_r_squared:.4f})')
print(
    f'T2* = {ramsey_fit[2]:.2f} +/- {ramsey_error[2]:.2f} us '
    f'(positive-detuning Ramsey fit, R^2 = {ramsey_r_squared:.4f})'
)
print(f'Ramsey oscillation frequency = {ramsey_fit[3]:.4f} MHz')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.25), constrained_layout=True)

t1_dense_us = np.linspace(t1_time_us.min(), t1_time_us.max(), 800)
axes[0].plot(
    t1_time_us, t1_state, linestyle='none', marker='o', markersize=2.7,
    markerfacecolor='#b2182b', markeredgecolor='#b2182b', markeredgewidth=0,
)
axes[0].plot(
    t1_dense_us,
    t1_decay(t1_dense_us, *t1_fit),
    color='#b2182b',
    linewidth=1.05,
    label=rf'$T_1={t1_fit[2]:.2f}\pm{t1_error[2]:.2f}\,\mu\mathrm{{s}}$',
)
axes[0].set_xlabel(r'Idle time ($\mu\mathrm{s}$)')
axes[0].set_ylabel(r'Excited-state probability, $P_e$')
axes[0].legend(loc='upper right', frameon=False)
axes[0].set_title('(a)', loc='left', pad=3, fontweight='bold')

ramsey_dense_us = np.linspace(ramsey_time_us.min(), ramsey_time_us.max(), 2_000)
axes[1].plot(
    ramsey_time_us, ramsey_state, linestyle='none', marker='o', markersize=2.4,
    markerfacecolor='#2166ac', markeredgecolor='#2166ac', markeredgewidth=0,
)
axes[1].plot(
    ramsey_dense_us,
    ramsey_decay(ramsey_dense_us, *ramsey_fit),
    color='#2166ac',
    linewidth=0.9,
    label=rf'$T_2^*={ramsey_fit[2]:.2f}\pm{ramsey_error[2]:.2f}\,\mu\mathrm{{s}}$',
)
axes[1].set_xlabel(r'Idle time ($\mu\mathrm{s}$)')
axes[1].legend(loc='upper right', frameon=False)
axes[1].set_title('(b)', loc='left', pad=3, fontweight='bold')

for axis in axes:
    axis.set_ylim(0.08, 1.02)
    axis.tick_params(which='both', direction='in', top=True, right=True)
    axis.grid(False)

saved_paths = save_figure(
    fig,
    '05_q1_t1_t2',
    variant=VARIANT,
    formats=('pdf', 'png', 'svg'),
    transparent=False,
)
saved_paths